# 04. Cycle Detection and Topological Sort

For **directed** graphs: can we order the vertices so that every edge goes from earlier to later? That is a **topological order**. It exists **only if there is no cycle** (the graph is a DAG).

**Learning order:** REAL LIFE → KAHN'S ALGORITHM → COURSE SCHEDULE → DFS CYCLE DETECTION → UNDIRECTED CYCLE → ALIEN DICTIONARY → PARALLEL COURSES → INTERVIEW THINKING

## 1. Real life

- **Course prerequisites:** to take Algorithms you need Data Structures. Find a valid order of courses.
- **Build systems / package managers:** compile `A` before `B` if `B` depends on `A`.
- **Getting dressed:** socks before shoes.

A **cycle** ("A needs B, B needs A") means no valid order (deadlock).

## 2. Kahn's algorithm (BFS with in-degrees)

1. Compute the **in-degree** (number of prerequisites) of every node.
2. Put all nodes with in-degree `0` in a queue (nothing blocks them).
3. Pop a node, add it to the order, and **reduce the in-degree of its neighbours**. A neighbour that reaches 0 is ready: enqueue it.
4. If the order contains all `n` nodes, it is valid. If fewer, **a cycle exists** (those nodes never reached in-degree 0).

O(V + E).

In [1]:
from collections import deque, defaultdict

def topological_order(n, prerequisites):              # prerequisites[i] = [course, must_take_first]
    graph, indegree = defaultdict(list), [0] * n
    for course, pre in prerequisites:
        graph[pre].append(course)
        indegree[course] += 1
    q = deque(i for i in range(n) if indegree[i] == 0)
    order = []
    while q:
        node = q.popleft()
        order.append(node)
        for nb in graph[node]:
            indegree[nb] -= 1
            if indegree[nb] == 0:
                q.append(nb)
    return order if len(order) == n else []           # empty = cycle

def can_finish(n, prerequisites):
    return len(topological_order(n, prerequisites)) == n

print(can_finish(2, [[1, 0]]), can_finish(2, [[1, 0], [0, 1]]))   # True False
print(topological_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]]))     # [0, 1, 2, 3]

True False
[0, 1, 2, 3]


## 3. DFS-based cycle detection (directed)

Use **3 states** per node: `0` = not visited, `1` = **in the current DFS path**, `2` = fully done. If we reach a node in state `1`, we found a **back edge** = cycle.

(A plain `visited` set is **not enough** for directed graphs: reaching an already finished node is fine, reaching a node on the current path is not.)

In [2]:
def has_cycle_directed(n, edges):
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
    state = [0] * n
    def dfs(u):
        state[u] = 1
        for v in graph[u]:
            if state[v] == 1:
                return True
            if state[v] == 0 and dfs(v):
                return True
        state[u] = 2
        return False
    return any(state[i] == 0 and dfs(i) for i in range(n))

print(has_cycle_directed(3, [(0, 1), (1, 2), (2, 0)]), has_cycle_directed(3, [(0, 1), (1, 2), (0, 2)]))   # True False

True False


## 4. Cycle in an undirected graph

In an undirected graph, every edge appears **twice**, so going back to the **parent** is not a cycle. During DFS, pass the `parent`. If you meet a **visited node that is not the parent**, there is a cycle.

(Alternative: Union-Find. If an edge connects two nodes already in the same set, it closes a cycle.)

In [3]:
def has_cycle_undirected(n, edges):
    graph = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
    visited = set()
    def dfs(node, parent):
        visited.add(node)
        for nb in graph[node]:
            if nb == parent:
                continue
            if nb in visited or dfs(nb, node):
                return True
        return False
    return any(i not in visited and dfs(i, -1) for i in range(n))

print(has_cycle_undirected(3, [(0, 1), (1, 2), (2, 0)]), has_cycle_undirected(3, [(0, 1), (1, 2)]))   # True False

True False


## 5. Alien Dictionary

Words are sorted in an unknown alphabet. Recover the letter order.

- Compare **adjacent words**. The **first differing letters** give an edge `a → b` ("a comes before b").
- If a longer word comes **before** its own prefix (`"abc"` before `"ab"`), the input is invalid.
- Topological sort of the letters. If a cycle exists, return `""`.

In [4]:
def alien_order(words):
    graph = {c: set() for w in words for c in w}
    indegree = {c: 0 for c in graph}
    for w1, w2 in zip(words, words[1:]):
        if len(w1) > len(w2) and w1.startswith(w2):
            return ""
        for a, b in zip(w1, w2):
            if a != b:
                if b not in graph[a]:
                    graph[a].add(b)
                    indegree[b] += 1
                break
    q = deque(c for c in indegree if indegree[c] == 0)
    order = []
    while q:
        c = q.popleft()
        order.append(c)
        for nb in graph[c]:
            indegree[nb] -= 1
            if indegree[nb] == 0:
                q.append(nb)
    return "".join(order) if len(order) == len(graph) else ""

print(alien_order(["wrt", "wrf", "er", "ett", "rftt"]), repr(alien_order(["abc", "ab"])))   # wertf ''

wertf ''


## 6. Parallel Courses (minimum semesters)

Take any number of courses per semester if their prerequisites are done. Minimum number of semesters = the **length of the longest path** in the DAG. With Kahn's algorithm, process **level by level**: each level is one semester. If not all courses are taken, return `-1`.

In [5]:
def minimum_semesters(n, relations):                  # relations: [prev, next], courses are 1..n
    graph, indegree = defaultdict(list), [0] * (n + 1)
    for a, b in relations:
        graph[a].append(b)
        indegree[b] += 1
    q = deque(i for i in range(1, n + 1) if indegree[i] == 0)
    semesters = taken = 0
    while q:
        semesters += 1
        for _ in range(len(q)):
            node = q.popleft()
            taken += 1
            for nb in graph[node]:
                indegree[nb] -= 1
                if indegree[nb] == 0:
                    q.append(nb)
    return semesters if taken == n else -1

print(minimum_semesters(3, [[1, 3], [2, 3]]), minimum_semesters(3, [[1, 2], [2, 3], [3, 1]]))   # 2 -1

2 -1


## 7. Interview thinking

**Signals:** "prerequisites", "dependencies", "order of tasks", "is it possible to finish", "build order", "alphabet order".

**Say:** "This is a dependency graph. A valid order exists iff it is a DAG. I'll run Kahn's algorithm (O(V + E)); if I cannot process all nodes there is a cycle."

**Common mistakes:** edge direction reversed (`[a, b]` means `b → a` in Course Schedule), using only a `visited` set for directed cycle detection, forgetting nodes with no edges, not handling the prefix-invalid case in Alien Dictionary.

## Quick Revision

- Topological order exists ⇔ DAG.
- Kahn: in-degrees, queue of zeros, reduce neighbours. `len(order) < n` ⇒ cycle.
- Directed cycle (DFS): 3 states, back edge to state 1.
- Undirected cycle: visited neighbour that is not the parent (or Union-Find).
- Levels of Kahn = semesters = longest path.